# Feature Heirarchy - What each CNN Layers Learn

**Theme:**

*“Early layers see edges. Deeper layers see concepts.”*

## Imports

*not much coding

In [1]:
import torch
import torch.nn as nn

torch.__version__

'2.8.0+cu129'

## Think layer by layer

CNN:
```powershell
Conv1 → ReLU → Pool
Conv2 → ReLU → Pool
FC → Output
```

Let’s break what each stage learns.

### What Conv1 Learns (Low-level features)


- First conv layer sees:
```nginx
Raw pixels
```

- So it learns filters that detect:

  - Edges

  - Lines

  - Corners

  - Light–dark transitions

📌 Answer:

- Why can’t the first layer learn digits directly?

- Why does it learn edges first?

### What Conv2 Learns (Mid-level features)


Conv2 doesn’t see pixels anymore.

It sees:

- Edges from Conv1


So it learns:

- Curves

- Intersections

- Small shapes

- Stroke patterns

📌 Answer:
- How is learning curves easier after edges are detected?

### What FC Layer Learns (High-level features)

By the time features reach FC:

- Model sees:

  - patterns of patterns


- So FC layer learns:

  - “Loop shape” → 0, 6, 9

  - “Vertical stroke” → 1

  - “Cross stroke” → 4

📌 Answer:

- Why is classification done at the FC layer instead of Conv layers?

## Why Deeper Networks Work Better

Imagine:

| Layers | Ability             |
| ------ | ------------------- |
| 1–2    | detect edges        |
| 3–5    | detect shapes       |
| 6–10   | detect object parts |
| 20+    | detect full objects |


This is called:

- Hierarchical feature learning

📌 Answer:

- Why does deeper = more expressive?

- What could go wrong if a network is too deep?

## Quick Inspection (Mini Code)

In [2]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, stride=1),  # 1(black&white)28x28x1 -> 28x28x32 *formula:- (width - filter + 2 x pdding)/stride + 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),                                # 28x28x32 -> 14x14x32

            nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=1), # 14x14x32 => 14x14x64
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)                                   # 14x14 => 7x7
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 512),
            nn.ReLU(),
            nn.Linear(512,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )
    
    def forward(self,x):
        x = self.conv(x)
        output = self.fc(x)
        return output 

model = CNN()

In [6]:
for name, layer in model.named_modules():
    print(name)


conv
conv.0
conv.1
conv.2
conv.3
conv.4
conv.5
fc
fc.0
fc.1
fc.2
fc.3
fc.4
fc.5


This is the structure hierarchy

📌 Answer:

- Why is a CNN basically a stack of feature extractors?

# Interaction

### Conv1 learning edges



**Misconception:** "CNN applies filter then starts detecting edges."

Explanation:

- The filters are NOT designed to detect edges. (In earlier filters are hand made although)
- The network learns filters that detect edges because edges are useful patterns to minimize the loss.

So:

- Initially filters are random.

After training, they become edge detectors automatically.

> This is a key deep learning insight:

Features are learned, not programmed.

### Conv2 learning curves

Conv2 combines outputs of Conv1 (edges) into more complex patterns like curves and strokes.

Because:
```java
A curve = combination of multiple edges

A corner = intersection of edges
```

---

### Why FC layer for classification


Precise explanation:

- Conv layers extract features.
- FC layer maps those features to class probabilities.

So:

- **Conv = feature extractor**

- **FC = decision maker**

---

### Depth


**Misconception** of deep networks:
- After detecting objects if we make our network more deep, it combines those object to create new objects/shapes

The real problems with very deep networks are:

- Vanishing gradients

- Overfitting

- Harder optimization

- More computation

Not exactly "creating new shapes".

Modern deep networks solve this using:

- Residual connections (ResNet)

- BatchNorm

---

### CNN = stack of feature extractors

### What actually happens in Conv1

`nn.Conv2d(1, 32, kernel_size=3)`

This means:

- Input channels = 1 (grayscale)

- Output channels = 32

So the network learns:

- 32 DIFFERENT filters

Each filter:

- Has its own weights

Detects a different pattern

Examples after training:
```bash
Filter 1 → vertical edges
Filter 2 → horizontal edges
Filter 3 → diagonal edges
Filter 4 → corners
Filter 5 → blobs
… up to 32 patterns
```
So:

- Same filter is reused across space
- But each channel has a different filter

This is the real meaning of weight sharing.

---

### Why same filter is applied across the image


One filter slides across the image.

So one filter learns:

- “Find vertical edges anywhere in the image”

This gives:

- **Translation invariance**

- Pattern detection independent of position


---

#### What happens in Conv2


`nn.Conv2d(32, 64, kernel_size=3)`

This means:

- Input = 32 feature maps

- Output = 64 filters

Now each new filter sees:

- All 32 previous feature maps together

So it learns:

- combinations of edges

- shapes

- curves

This is how hierarchy forms.

---

#### Visual mental model

Think of Conv1 as:

- 32 pattern detectors

Conv2 becomes:

- 64 pattern combiners

And deeper layers become:

- complex pattern recognizers
> One-sentence explanation (remember this)

Each filter learns a different feature, and the same filter is reused across the image to detect that feature anywhere.